<a href="https://colab.research.google.com/github/codeKrantz/UserPDBModeling-PocketDownload/blob/main/UserModeling%2BPocketDownload.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# User PDB Modeling and Enzyme Pocket Download

This Noteboolk provides tools for:
*   Modeling an enzyme using mol view
*   Saving the pocket of the protein
* Can use a PDB ID
* Can use in-house PDB files


## Setup and Dependencies

Install required packages and import libraries. Run this cell first.
WARNING: you may need to restart your session for the interpro to be correctly installed. Ignore the pop up for third party widgets.

In [ ]:
#@title InterProScan setup
!git clone https://github.com/ebi-interpro/docs.git
%cd docs
#this step gernally wants a new runtime session after the first attempt to install
!pip install -r requirements.txt
!pip install -q molview
!pip -q install biopython nglview pandas requests
!pip -q install biopython nglview pandas requests


#Imports after Installs
import requests
import pandas as pd
import molview as mv
from Bio.PDB import MMCIFParser
from Bio.PDB import PDBParser, NeighborSearch, Select, PDBIO
import nglview as nv
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files


## 🖥️ Interactive Interface

**Run this cell to display the display interface.**

The interface will appear below the code box.

In [ ]:
 #@title Pocket scoop
# If needed in a fresh Colab runtime, uncomment this line:
%pip -q install biopython molview ipywidgets pandas requests

import os
import numpy as np
import pandas as pd
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
import molview as mv
from google.colab import files
from Bio.PDB import PDBParser, Select, PDBIO

# ---------------- SETTINGS ----------------
ACTIVE_SITE_RADIUS_ANGSTROM = 5.0
EXCLUDE_WATER = True

# ---------------- STATE ----------------
latest_output_file = {"path": None}
uploaded_pdb_file = {"path": None, "label": None}
latest_full_pdb = {"path": None, "label": None}

# ---------------- ACTIVE-SITE / POCKET LOGIC ----------------
def is_polymer_residue(residue) -> bool:
    """True for standard polymer residues (not hetero, not water)."""
    return residue.id[0] == " "

def extract_protein_atoms(structure):
    """Collect all polymer atoms as tuples:
    (model_id, chain_id, residue_obj, atom_obj, atom_coord)
    """
    protein_atoms = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if not is_polymer_residue(residue):
                    continue
                for atom in residue.get_atoms():
                    protein_atoms.append((model.id, chain.id, residue, atom, atom.coord))
    return protein_atoms

def extract_hetero_atoms_from_heteros(heteros):
    """Convert hetero residue list into hetero atom list."""
    hetero_atoms = []
    for model_id, chain_id, residue in heteros:
        hetflag = residue.id[0]
        if EXCLUDE_WATER and hetflag == "W":
            continue
        for atom in residue.get_atoms():
            hetero_atoms.append((model_id, chain_id, residue, atom, atom.coord))
    return hetero_atoms

def active_site_from_ligands(structure, heteros, radius=5.0):
    """
    Returns:
      active_residues_df: unique polymer residues within radius Å of any hetero atom
      contacts_df: detailed residue↔ligand contacts
    """
    protein_atoms = extract_protein_atoms(structure)
    hetero_atoms = extract_hetero_atoms_from_heteros(heteros)

    if len(hetero_atoms) == 0:
        raise ValueError("No heteroatoms found after filtering.")

    prot_coords = np.array([p[4] for p in protein_atoms], dtype=float)
    r2 = float(radius) ** 2

    active_residue_keys = set()
    contact_pairs = set()

    for (h_model, h_chain, h_res, h_atom, h_xyz) in hetero_atoms:
        d2 = np.sum((prot_coords - h_xyz) ** 2, axis=1)
        close_idx = np.where(d2 <= r2)[0]

        if close_idx.size == 0:
            continue

        h_hetflag, h_resseq, h_icode = h_res.id
        ligand_key = (h_model, h_chain, h_res.get_resname(), h_hetflag, h_resseq, h_icode)

        for i in close_idx:
            p_model, p_chain, p_res, p_atom, p_xyz = protein_atoms[i]
            p_hetflag, p_resseq, p_icode = p_res.id

            prot_key = (p_model, p_chain, p_res.get_resname(), p_resseq, p_icode)
            active_residue_keys.add(prot_key)
            contact_pairs.add((prot_key, ligand_key))

    active_residues_df = pd.DataFrame(
        list(active_residue_keys),
        columns=["model", "chain", "resname", "resseq", "icode"]
    )

    if active_residues_df.empty:
        active_residues_df = pd.DataFrame(columns=["model", "chain", "resname", "resseq", "icode"])
    else:
        active_residues_df = (
            active_residues_df
            .sort_values(["model", "chain", "resseq", "icode"])
            .reset_index(drop=True)
        )

    contacts_df = pd.DataFrame(
        [(pk[0], pk[1], pk[2], pk[3], pk[4], lk[0], lk[1], lk[2], lk[3], lk[4], lk[5])
         for pk, lk in contact_pairs],
        columns=[
            "prot_model","prot_chain","prot_resname","prot_resseq","prot_icode",
            "lig_model","lig_chain","lig_resname","lig_hetflag","lig_resseq","lig_icode"
        ]
    )

    if contacts_df.empty:
        contacts_df = pd.DataFrame(columns=[
            "prot_model","prot_chain","prot_resname","prot_resseq","prot_icode",
            "lig_model","lig_chain","lig_resname","lig_hetflag","lig_resseq","lig_icode"
        ])
    else:
        contacts_df = (
            contacts_df
            .sort_values(["prot_model","prot_chain","prot_resseq","lig_resname","lig_chain","lig_resseq"])
            .reset_index(drop=True)
        )

    return active_residues_df, contacts_df

class SelectResidues(Select):
    def __init__(self, select_residues):
        self.target_residues = set(select_residues)

    def accept_residue(self, residue):
        return residue in self.target_residues

def save_pocket(structure, residues, code, resname):
    """Save selected residues as a pocket PDB."""
    io = PDBIO()
    io.set_structure(structure)

    safe_code = str(code).replace(" ", "_")
    safe_resname = str(resname).replace(" ", "_")
    filename = f"./pockets/{safe_code}_{safe_resname}.pdb"

    os.makedirs(os.path.dirname(filename), exist_ok=True)
    io.save(filename, SelectResidues(residues))

    print(f"[SAVED] {filename} with {len(residues)} residues")
    return filename

# ---------------- INPUT HELPERS ----------------
def pdb_exists(pdb_id):
    pdb_id = pdb_id.strip().upper()
    if not pdb_id:
        return False

    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(url, timeout=30)
    return response.status_code == 200 and len(response.text.strip()) > 0

def download_pdb_by_id(pdb_id):
    pdb_id = pdb_id.strip().upper()
    print(f"Processing public PDB ID: {pdb_id}...")

    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(url, timeout=60)

    if response.status_code != 200 or not response.text.strip():
        print("[ERROR] Failed to download PDB file.")
        return None, None, None

    output_path = f"/content/{pdb_id}.pdb"
    with open(output_path, "w") as f:
        f.write(response.text)

    print(f"[SAVED] {output_path}")
    return output_path, pdb_id, response.text

def upload_local_pdb():
    uploaded = files.upload()

    if not uploaded:
        print("[ERROR] No file uploaded.")
        return None, None, None

    original_name = list(uploaded.keys())[0]

    if not original_name.lower().endswith(".pdb"):
        print("[ERROR] Please upload a .pdb file only.")
        return None, None, None

    label = os.path.splitext(original_name)[0]
    stable_path = "/content/current_upload.pdb"

    with open(stable_path, "wb") as f:
        f.write(uploaded[original_name])

    with open(stable_path, "r") as f:
        pdb_text = f.read()

    uploaded_pdb_file["path"] = stable_path
    uploaded_pdb_file["label"] = label

    print(f"[UPLOADED] {original_name}")
    print(f"[SAVED AS] {stable_path}")

    return stable_path, label, pdb_text

# ---------------- MAIN PROCESSING ----------------


def process_structure(file_path, structure_label, pdb_text):
    if not file_path or not os.path.exists(file_path):
        print("[ERROR] File does not exist.")
        return

    latest_full_pdb["path"] = file_path
    latest_full_pdb["label"] = structure_label
    latest_output_file["path"] = None
    download_button.disabled = True

    print(f"[READY] Visualizing {structure_label}...")
    v = mv.view(width=600, height=500, panel=True)
    v.addModel(pdb_text, name=structure_label)
    v.setColorMode("custom")
    v.show()

    # Parse structure
    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(structure_label, file_path)
    except Exception as e:
        print(f"[ERROR] Failed to parse structure: {e}")
        return

    # Extract hetero residues (non-polymer, non-water)
    heteros = []
    for model in structure:
        for chain in model:
            for residue in chain:
                hetflag = residue.id[0]
                if hetflag != " " and hetflag != "W":
                    heteros.append((model.id, chain.id, residue))

    if not heteros:
        print("[ERROR] No hetero residues found, so no ligand-based pocket could be saved.")
        return

    # Find active site residues near ligands
    try:
        active_residues_df, contacts_df = active_site_from_ligands(
            structure, heteros, radius=ACTIVE_SITE_RADIUS_ANGSTROM
        )
    except Exception as e:
        print(f"[ERROR] Failed to identify active site: {e}")
        return

    if active_residues_df.empty:
        print("[ERROR] No active-site residues found.")
        return

    # Convert dataframe rows back to actual Bio.PDB residue objects
    active_residue_objects = []
    wanted_keys = {
        (row["model"], row["chain"], row["resname"], int(row["resseq"]), row["icode"])
        for _, row in active_residues_df.iterrows()
    }

    for model in structure:
        for chain in model:
            for residue in chain:
                key = (
                    model.id,
                    chain.id,
                    residue.get_resname(),
                    residue.id[1],
                    residue.id[2]
                )
                if key in wanted_keys:
                    active_residue_objects.append(residue)

    active_residue_objects = list(set(active_residue_objects))

    if not active_residue_objects:
        print("[ERROR] Could not map active-site dataframe back to residue objects.")
        return

    ligand_name = contacts_df["lig_resname"].iloc[0]
    pocket_path = save_pocket(structure, active_residue_objects, structure_label, ligand_name)

    if os.path.exists(pocket_path):
        latest_output_file["path"] = pocket_path
        download_button.disabled = False
        print(f"[READY] Pocket file saved at: {pocket_path}")
        print(f"[INFO] Active-site residues found: {len(active_residue_objects)}")
        print(f"[INFO] Unique ligand residues involved: {contacts_df[['lig_chain','lig_resname','lig_resseq']].drop_duplicates().shape[0]}")
    else:
        print("[ERROR] Pocket file was not created.")

# ---------------- UI ----------------
input_mode = widgets.ToggleButtons(
    options=["PDB ID", "Upload File"],
    description="Input Mode:"
)

pdb_input = widgets.Text(
    value="",
    placeholder="Enter PDB ID",
    description="PDB ID:"
)

upload_button = widgets.Button(
    description="Upload PDB File",
    button_style="warning"
)

run_button = widgets.Button(
    description="Run",
    button_style="success"
)

download_button = widgets.Button(
    description="Download Pocket",
    button_style="info",
    disabled=True
)

output = widgets.Output()

def update_input_visibility(change=None):
    if input_mode.value == "PDB ID":
        pdb_input.layout.display = "block"
        upload_button.layout.display = "none"
    else:
        pdb_input.layout.display = "none"
        upload_button.layout.display = "block"

def on_upload_clicked(b):
    with output:
        clear_output()
        upload_local_pdb()

def on_download_clicked(b):
    path = latest_output_file["path"]
    if path and os.path.exists(path):
        files.download(path)
    else:
        with output:
            print("No pocket file ready to download yet.")

def on_run_clicked(b):
    with output:
        clear_output()

        if input_mode.value == "PDB ID":
            pdb_id = pdb_input.value.strip()

            if not pdb_id:
                print("[ERROR] Please enter a PDB ID.")
                return

            if not pdb_exists(pdb_id):
                print("[ERROR] PDB does NOT exist.")
                return

            file_path, structure_label, pdb_text = download_pdb_by_id(pdb_id)
            if file_path:
                process_structure(file_path, structure_label, pdb_text)

        elif input_mode.value == "Upload File":
            file_path = uploaded_pdb_file["path"]
            structure_label = uploaded_pdb_file["label"]

            if not file_path:
                print("[ERROR] Please upload a .pdb file first.")
                return

            with open(file_path, "r") as f:
                pdb_text = f.read()

            process_structure(file_path, structure_label, pdb_text)

input_mode.observe(update_input_visibility, names="value")
upload_button.on_click(on_upload_clicked)
download_button.on_click(on_download_clicked)
run_button.on_click(on_run_clicked)

display(input_mode)
display(pdb_input)
display(upload_button)
display(run_button)
display(download_button)
display(output)

update_input_visibility()

In [ ]:
#@title Visualize Last Generated Active Site Pocket

import os
import molview as mv

# Check if a first shell pocket has been generated
pocket_path = latest_output_file["path"]

if pocket_path is None:
    print("[ERROR] No active-site pocket has been generated yet.")
    print("Run the interface cell first.")

elif not os.path.exists(pocket_path):
    print(f"[ERROR] Pocket file not found: {pocket_path}")

else:
    print(f"[LOADING] {pocket_path}")

    # Create viewer
    v = mv.view(width=700, height=550, panel=True)

    # Load most recent generated pocket
    with open(pocket_path, "r") as f:
        pdb_data = f.read()

    v.addModel(pdb_data, name="Active Site Pocket")

    # Nice coloring
    v.setColorMode("element")

    # Show viewer
    v.show()

In [ ]:
#@title Extract all shells
%pip -q install biopython molview pandas numpy requests

import os
import numpy as np
import pandas as pd
import molview as mv
from Bio.PDB import PDBParser, NeighborSearch

# -------- parameters --------
FULL_PDB_PATH = latest_full_pdb["path"]

if not FULL_PDB_PATH or not os.path.exists(FULL_PDB_PATH):
    raise ValueError("No full PDB available yet. Click Run in the UI first (PDB ID or Upload).")
STRUCTURE_LABEL = "5TIC"
CHAIN_ID = "A"

LIGAND_RADIUS = 5.0         # shell 1: within this Å of ligand atoms
SHELL_STEP_RADIUS = 3.0     # shell 2+: within this Å of previous shell atoms
EXCLUDE_WATER = True
MAX_SHELLS = 20

# -------- 1-letter mapping --------
AA3_TO_1 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
    "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
    "THR":"T","TRP":"W","TYR":"Y","VAL":"V",
    "MSE":"M","SEC":"U","PYL":"O","ASX":"B","GLX":"Z","XAA":"X"
}

def res_to_key(res):
    hetflag, resseq, icode = res.id
    return (int(resseq), str(icode).strip() or "", res.get_resname().strip())

def key_to_label(key):
    resseq, icode, resname = key
    aa1 = AA3_TO_1.get(resname.upper(), "X")
    pos = f"{resseq}{icode}" if icode else f"{resseq}"
    return f"{aa1}{pos}"

def is_standard_residue(res):
    return res.id[0] == " "

def is_water(res):
    return res.id[0] == "W" or res.get_resname().upper() in {"HOH", "WAT"}

# -------- load structure --------
parser = PDBParser(QUIET=True)
structure = parser.get_structure(STRUCTURE_LABEL, FULL_PDB_PATH)
model = structure[0]

if CHAIN_ID not in model:
    raise ValueError(f"Chain {CHAIN_ID} not found. Available: {list(model.child_dict.keys())}")

chain = model[CHAIN_ID]

# NeighborSearch over all atoms in the full structure
all_atoms = list(structure.get_atoms())
ns = NeighborSearch(all_atoms)

# -------- find ligand residues (hetero residues) in this chain --------
ligand_residues = []
for res in chain:
    if res.id[0] == " ":
        continue
    if EXCLUDE_WATER and is_water(res):
        continue
    ligand_residues.append(res)

if not ligand_residues:
    # helpful debug info
    hets = [(res.get_resname(), res.id) for res in chain if res.id[0] != " "]
    raise ValueError(
        "No ligand/hetero residues found in this chain.\n"
        f"Hetero residues seen in chain {CHAIN_ID}: {hets}\n"
        "Tip: ensure you're using the FULL PDB (not the pocket-only PDB), "
        "and/or check if the ligand is in a different chain."
    )

ligand_atoms = [a for lr in ligand_residues for a in lr.get_atoms()]

# -------- shell 1 (near ligand) --------
shells = {}      # residue_key -> shell_number
shell_order = [] # list[set[residue_key]]

shell1 = set()
for latom in ligand_atoms:
    for nb_atom in ns.search(latom.coord, LIGAND_RADIUS):
        nb_res = nb_atom.get_parent()

        # only polymer residues in selected chain
        if nb_res.get_parent().id != CHAIN_ID:
            continue
        if not is_standard_residue(nb_res):
            continue

        shell1.add(res_to_key(nb_res))

if not shell1:
    raise ValueError(f"No protein residues within {LIGAND_RADIUS} Å of ligand atoms.")

for k in shell1:
    shells[k] = 1
shell_order.append(shell1)

# -------- shell 2..N --------
current_shell = 1
while current_shell < MAX_SHELLS:
    prev_keys = shell_order[current_shell - 1]
    new_keys = set()

    prev_residues = []
    for res in chain:
        if is_standard_residue(res) and res_to_key(res) in prev_keys:
            prev_residues.append(res)

    for res in prev_residues:
        for atom in res.get_atoms():
            for nb_atom in ns.search(atom.coord, SHELL_STEP_RADIUS):
                nb_res = nb_atom.get_parent()

                if nb_res.get_parent().id != CHAIN_ID:
                    continue
                if not is_standard_residue(nb_res):
                    continue

                k = res_to_key(nb_res)
                if k in shells:
                    continue
                new_keys.add(k)

    if not new_keys:
        break

    current_shell += 1
    for k in new_keys:
        shells[k] = current_shell
    shell_order.append(new_keys)

# -------- table --------
rows = []
for sh_idx, keyset in enumerate(shell_order, start=1):
    labels = sorted([key_to_label(k) for k in keyset],
                    key=lambda x: (int("".join(filter(str.isdigit, x)) or 0), x))
    rows.append({"shell": sh_idx, "n_residues": len(labels), "residues": ", ".join(labels)})

df_shells = pd.DataFrame(rows, columns=["shell", "n_residues", "residues"])
df_shells

In [ ]:
#@title Visualize all shells

from Bio.PDB import PDBIO, Select
from io import StringIO
import molview as mv

# Requires from previous "Extract all shells" cell:
# structure, chain, shell_order, res_to_key, FULL_PDB_PATH, STRUCTURE_LABEL

# --- palette: one color per shell (repeats if needed) ---
palette = [
    "#e41a1c",  # red
    "#ff7f00",  # orange
    "#ffd92f",  # yellow
    "#4daf4a",  # green
    "#00c5cd",  # cyan
    "#377eb8",  # blue
    "#984ea3",  # purple
    "#f781bf",  # pink
    "#a65628",  # brown
    "#999999",  # gray
]

# Map residue keys to residue objects (polymer only) for this chain
res_obj_by_key = {}
for res in chain:
    if res.id[0] == " ":
        res_obj_by_key[res_to_key(res)] = res

class OnlyResidues(Select):
    def __init__(self, allowed):
        self.allowed = set(allowed)
    def accept_residue(self, residue):
        return residue in self.allowed

def pdb_text_for_residues(structure, residues):
    io = PDBIO()
    io.set_structure(structure)
    buf = StringIO()
    io.save(buf, select=OnlyResidues(residues))
    return buf.getvalue()

# Load full PDB text for the background model
with open(FULL_PDB_PATH, "r") as f:
    full_pdb_text = f.read()

v = mv.view(width=900, height=650, panel=True)

# Model 0: full structure (background)
v.addModel(full_pdb_text, name=f"{STRUCTURE_LABEL} full")

# Add one model per shell containing only that shell’s residues
shell_model_indices = []  # (shell_idx, model_index)
for shell_idx, keyset in enumerate(shell_order, start=1):
    residues = [res_obj_by_key[k] for k in keyset if k in res_obj_by_key]
    if not residues:
        continue
    shell_pdb = pdb_text_for_residues(structure, residues)
    v.addModel(shell_pdb, name=f"shell {shell_idx}")
    # model index is appended in order: full is 0, first shell is 1, etc.
    shell_model_indices.append((shell_idx, len(shell_model_indices) + 1))

# Representation:
# - full structure: cartoon (gray)
# - shells: sticks (colored)
v.setStyle({"cartoon": {"color": "#d0d0d0"}})   # applies globally; shells will still show sticks if enabled too
v.setStyle({"stick": {}})                      # enable sticks for shell models (molview toggles stick support)

# Switch to custom color mode.
# We'll set per-model colors via color_params if supported.
v.setColorMode("custom")

# Try to set per-model colors using color_params (works on many molview builds).
# If your build ignores this, tell me what v.color_params contains after show().
try:
    # Make sure color_params exists and is a dict-like
    if isinstance(v.color_params, dict):
        # color_params format varies by version; commonly includes 'models' or similar.
        # We'll set a simple mapping if supported.
        v.color_params["model_colors"] = {0: "#d0d0d0"}  # background
        for shell_idx, model_idx in shell_model_indices:
            v.color_params["model_colors"][model_idx] = palette[(shell_idx - 1) % len(palette)]
except Exception as e:
    print("Note: could not set per-model colors via color_params:", e)

v.show()

In [ ]:
#@title All shells
from Bio.PDB import PDBIO, Select
from io import StringIO
import molview as mv



# --- helper: turn a set of residue keys into a PDB string ---
res_obj_by_key = {}
for res in chain:
    if res.id[0] == " ":
        res_obj_by_key[res_to_key(res)] = res

class OnlyResidues(Select):
    def __init__(self, allowed):
        self.allowed = set(allowed)
    def accept_residue(self, residue):
        return residue in self.allowed

def pdb_text_for_residues(structure, residues):
    io = PDBIO()
    io.set_structure(structure)
    buf = StringIO()
    io.save(buf, select=OnlyResidues(residues))
    return buf.getvalue()

# --- read full PDB ---
with open(FULL_PDB_PATH, "r") as f:
    full_pdb = f.read()

v = mv.view(width=900, height=650, panel=True)

# Model 0: full structure
v.addModel(full_pdb, name=f"{STRUCTURE_LABEL} (full)")

# Models 1..N: shells
for shell_idx, keyset in enumerate(shell_order, start=1):
    residues = [res_obj_by_key[k] for k in keyset if k in res_obj_by_key]
    if not residues:
        continue
    shell_pdb = pdb_text_for_residues(structure, residues)
    v.addModel(shell_pdb, name=f"shell {shell_idx}")

# Global styles (molview limitation: applies to all models)
# Cartoon gives context; sticks shows shells.
v.setLayout('grid')
v.setStyle({"cartoon": {}})
v.setIllustrativeStyle(True)
v.setStyle({"stick": {}})

v.setColorMode("element")


v.show()